# Host SigLIP2 Embedding Service

Notebook này host SigLIP2 dưới dạng OpenAI-compatible endpoint để search embedding. Model/search metadata được đọc trực tiếp từ `configs/model_registry.yaml`, entry `siglip2_so400m16_384_webli_openclip_1152_v1`, để tránh dùng nhầm SigLIP2 checkpoint/vector space khác.

## Contract

- Source of truth: `configs/model_registry.yaml`.
- Model id gửi vào `/v1/embeddings`: lấy từ field `model`.
- OpenCLIP model/pretrained để load weights: lấy từ `openclip_model` và `openclip_pretrained`.
- Collection search: lấy từ `collection`.
- Expected dim: lấy từ `dim`; notebook sẽ fail nếu vector trả về không khớp.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "configs" / "model_registry.yaml").exists():
            return candidate
    raise FileNotFoundError("Cannot find repo root containing configs/model_registry.yaml")


REPO_ROOT = find_repo_root()
BACKEND_ROOT = REPO_ROOT / "apps" / "backend"
for path in (REPO_ROOT, BACKEND_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print("Repo root:", REPO_ROOT)


In [ ]:
# Chỉ bật cell này khi môi trường chưa có dependency cần thiết.
INSTALL_DEPS = False

if INSTALL_DEPS:
    import subprocess

    requirements = REPO_ROOT / "apps" / "backend" / "requirements-embedding-service.txt"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements)])


In [ ]:
import yaml


SIGLIP2_MODEL_KEY = "siglip2_so400m16_384_webli_openclip_1152_v1"
REGISTRY_PATH = REPO_ROOT / "configs" / "model_registry.yaml"

with REGISTRY_PATH.open("r", encoding="utf-8") as handle:
    registry = yaml.safe_load(handle) or {}

embedders = registry.get("embedders") or {}
if SIGLIP2_MODEL_KEY not in embedders:
    raise KeyError(f"Missing {SIGLIP2_MODEL_KEY!r} in {REGISTRY_PATH}")

cfg = embedders[SIGLIP2_MODEL_KEY]
required_fields = ["model", "openclip_model", "openclip_pretrained", "dim", "collection"]
missing = [field for field in required_fields if not str(cfg.get(field, "")).strip()]
if missing:
    raise ValueError(f"SigLIP2 registry entry is missing fields: {missing}")

model_id = str(cfg["model"]).strip()
openclip_model = str(cfg["openclip_model"]).strip()
openclip_pretrained = str(cfg["openclip_pretrained"]).strip()
if "siglip2" not in f"{model_id} {openclip_model}".lower():
    raise ValueError(f"Configured model does not look like SigLIP2: {model_id} / {openclip_model}")

MODEL_INFO = {
    "model_key": SIGLIP2_MODEL_KEY,
    "provider": str(cfg.get("provider", "")).strip(),
    "model_id": model_id,
    "openclip_model": openclip_model,
    "openclip_pretrained": openclip_pretrained,
    "device": str(cfg.get("openclip_device") or cfg.get("device") or "auto").strip(),
    "max_batch_size": int(cfg.get("openclip_max_batch") or 16),
    "expected_dim": int(cfg["dim"]),
    "l2_normalize": bool(cfg.get("l2_normalize", True)),
    "base_url": str(cfg.get("base_url") or "http://127.0.0.1:8003/v1").strip().rstrip("/"),
    "collection": str(cfg["collection"]).strip(),
    "extractor_version": str(cfg.get("extractor_version") or "").strip(),
}

print(json.dumps(MODEL_INFO, indent=2, ensure_ascii=False))


In [ ]:
import math

import open_clip
import torch
import torch.nn.functional as F


def resolve_device(value: str) -> str:
    requested = str(value or "").strip().lower()
    if requested in {"", "auto"}:
        return "cuda" if torch.cuda.is_available() else "cpu"
    return str(value)


DEVICE = resolve_device(MODEL_INFO["device"])
print("Loading:", MODEL_INFO["openclip_model"], "| pretrained:", MODEL_INFO["openclip_pretrained"], "| device:", DEVICE)

clip_model, _, preprocess = open_clip.create_model_and_transforms(
    model_name=MODEL_INFO["openclip_model"],
    pretrained=MODEL_INFO["openclip_pretrained"],
    device=DEVICE,
)
tokenizer = open_clip.get_tokenizer(MODEL_INFO["openclip_model"])
clip_model.eval()

with torch.inference_mode():
    probe_tokens = tokenizer(["a person wearing a red shirt"]).to(DEVICE)
    probe = clip_model.encode_text(probe_tokens).to(torch.float32)
    if MODEL_INFO["l2_normalize"]:
        probe = F.normalize(probe, dim=-1)

actual_dim = int(probe.shape[-1])
probe_norm = float(torch.linalg.vector_norm(probe[0]).detach().cpu())
if actual_dim != MODEL_INFO["expected_dim"]:
    raise ValueError(f"Embedding dimension mismatch: expected {MODEL_INFO['expected_dim']}, got {actual_dim}")

print({"dim": actual_dim, "norm": round(probe_norm, 6), "model_id": MODEL_INFO["model_id"]})


In [ ]:
from typing import Any

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel


class EmbeddingRequest(BaseModel):
    model: str | None = None
    input: str | list[str]


def as_list(value: str | list[str]) -> list[str]:
    if isinstance(value, str):
        return [value]
    return [str(item) for item in value]


app = FastAPI(title="SigLIP2 OpenCLIP Embedding Service", version="1.0.0")


@app.get("/healthz")
def healthz() -> dict[str, Any]:
    return {
        "status": "ok",
        "model_key": MODEL_INFO["model_key"],
        "model_id": MODEL_INFO["model_id"],
        "openclip_model": MODEL_INFO["openclip_model"],
        "openclip_pretrained": MODEL_INFO["openclip_pretrained"],
        "device": DEVICE,
        "dim": MODEL_INFO["expected_dim"],
        "collection": MODEL_INFO["collection"],
        "l2_normalize": MODEL_INFO["l2_normalize"],
    }


@app.get("/v1/models")
def list_models() -> dict[str, Any]:
    return {
        "object": "list",
        "data": [{"id": MODEL_INFO["model_id"], "object": "model", "owned_by": "local"}],
    }


@app.post("/v1/embeddings")
def embeddings(request: EmbeddingRequest) -> dict[str, Any]:
    texts = as_list(request.input)
    if not texts or any(not text.strip() for text in texts):
        raise HTTPException(status_code=400, detail="input must not be empty")
    if len(texts) > MODEL_INFO["max_batch_size"]:
        raise HTTPException(status_code=400, detail=f"batch too large: {len(texts)} > {MODEL_INFO['max_batch_size']}")
    if request.model and request.model != MODEL_INFO["model_id"]:
        raise HTTPException(status_code=400, detail=f"model mismatch: requested {request.model!r}, available {MODEL_INFO['model_id']!r}")

    with torch.inference_mode():
        tokens = tokenizer(texts).to(DEVICE)
        vectors = clip_model.encode_text(tokens).to(torch.float32)
        if MODEL_INFO["l2_normalize"]:
            vectors = F.normalize(vectors, dim=-1)

    data = []
    token_count = 0
    for index, vector in enumerate(vectors):
        values = vector.detach().cpu().tolist()
        if len(values) != MODEL_INFO["expected_dim"]:
            raise HTTPException(status_code=500, detail=f"embedding dim mismatch: {len(values)}")
        data.append({"object": "embedding", "index": index, "embedding": values})
        token_count += int(tokens[index].numel())

    return {
        "object": "list",
        "data": data,
        "model": MODEL_INFO["model_id"],
        "usage": {"prompt_tokens": token_count, "total_tokens": token_count},
    }


In [ ]:
import threading
import time
from urllib.parse import urlparse

import httpx
import uvicorn


def stop_existing_server() -> None:
    server = globals().get("SIGLIP2_SERVER")
    thread = globals().get("SIGLIP2_THREAD")
    if server is not None:
        server.should_exit = True
    if thread is not None and thread.is_alive():
        thread.join(timeout=5)


parsed = urlparse(MODEL_INFO["base_url"])
HOST = os.getenv("SIGLIP2_EMBED_HOST") or parsed.hostname or "127.0.0.1"
PORT = int(os.getenv("SIGLIP2_EMBED_PORT") or parsed.port or 8003)
CLIENT_HOST = "127.0.0.1" if HOST in {"0.0.0.0", "::"} else HOST
CLIENT_BASE_URL = f"http://{CLIENT_HOST}:{PORT}/v1"
HEALTH_URL = f"http://{CLIENT_HOST}:{PORT}/healthz"
os.environ["SIGLIP2_EMBEDDING_BASE_URL"] = CLIENT_BASE_URL
os.environ["EMBEDDING_BASE_URL"] = CLIENT_BASE_URL

stop_existing_server()

SIGLIP2_SERVER = uvicorn.Server(uvicorn.Config(app, host=HOST, port=PORT, log_level="info"))
SIGLIP2_THREAD = threading.Thread(target=SIGLIP2_SERVER.run, daemon=True)
SIGLIP2_THREAD.start()

last_error = None
for _ in range(60):
    try:
        response = httpx.get(HEALTH_URL, timeout=2.0)
        if response.status_code == 200:
            print("Serving SigLIP2 embeddings at:", CLIENT_BASE_URL)
            print(json.dumps(response.json(), indent=2, ensure_ascii=False))
            break
    except Exception as exc:
        last_error = exc
    time.sleep(0.5)
else:
    raise RuntimeError(f"Embedding service did not become ready: {last_error}")


In [ ]:
def vector_norm(values: list[float]) -> float:
    return math.sqrt(sum(float(value) * float(value) for value in values))


with httpx.Client(timeout=30.0) as client:
    models = client.get(f"{CLIENT_BASE_URL}/models").json()
    available = {item.get("id") for item in models.get("data", [])}
    if MODEL_INFO["model_id"] not in available:
        raise RuntimeError({"expected": MODEL_INFO["model_id"], "available": sorted(available)})

    payload = {"model": MODEL_INFO["model_id"], "input": "a person wearing a red shirt in a crowded street"}
    body = client.post(f"{CLIENT_BASE_URL}/embeddings", json=payload).json()

vector = body["data"][0]["embedding"]
summary = {
    "ok": True,
    "model": body["model"],
    "dim": len(vector),
    "norm": round(vector_norm(vector), 6),
    "base_url": CLIENT_BASE_URL,
    "collection": MODEL_INFO["collection"],
}
if summary["dim"] != MODEL_INFO["expected_dim"]:
    raise RuntimeError(summary)
if MODEL_INFO["l2_normalize"] and abs(summary["norm"] - 1.0) > 1e-2:
    raise RuntimeError(summary)

print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:
import subprocess


SAMPLE_QUERY = "a person wearing a red shirt in a crowded street"
TOP_K = 20


def build_milvus_search_command(query: str, top_k: int = TOP_K) -> list[str]:
    return [
        sys.executable,
        str(REPO_ROOT / "apps" / "backend" / "scripts" / "search_milvus_siglip2.py"),
        "--query",
        query,
        "--collection",
        MODEL_INFO["collection"],
        "--model-name",
        MODEL_INFO["model_id"],
        "--embedding-base-url",
        CLIENT_BASE_URL,
        "--top-k",
        str(top_k),
    ]


def build_gcs_search_command(query: str, batches: str, top_k: int = TOP_K) -> list[str]:
    command = [
        sys.executable,
        str(REPO_ROOT / "apps" / "backend" / "scripts" / "search_gcs_siglip2_vectors.py"),
        "--query",
        query,
        "--batches",
        batches,
        "--model-name",
        MODEL_INFO["model_id"],
        "--embedding-base-url",
        CLIENT_BASE_URL,
        "--top-k",
        str(top_k),
    ]
    if MODEL_INFO["extractor_version"]:
        command.extend(["--extractor-version", MODEL_INFO["extractor_version"]])
    return command


print("Milvus search command:")
print(subprocess.list2cmdline(build_milvus_search_command(SAMPLE_QUERY)))
print("\nGCS scan search command:")
print(subprocess.list2cmdline(build_gcs_search_command(SAMPLE_QUERY, batches="L21,L22")))


In [ ]:
RUN_MILVUS_SEARCH = False
QUERY = SAMPLE_QUERY

if RUN_MILVUS_SEARCH:
    completed = subprocess.run(
        build_milvus_search_command(QUERY, top_k=TOP_K),
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
        check=True,
    )
    print(completed.stdout)


In [ ]:
# Chạy cell này khi muốn tắt service trong notebook kernel hiện tại.
# stop_existing_server()
# print("SigLIP2 embedding service stopped")
